In [6]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [ ]:
db_config = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": "get_password",  # <- 여기에 실제 비밀번호(영문/숫자/특수문자 등) 입력
    "database": "crime_project",
    "charset": "utf8mb4",
    "autocommit": False,
}

In [ ]:
# MariaDB에 적재하는 코드

import pandas as pd
import pymysql

crime_path = "경찰청_범죄 발생 지역별 통계_20251231.csv"
cctv_path = "CCTV정보.csv"

db_config = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "get_password",  # 실제 DB 비밀번호로 수정
    "database": "crime_project",
    "charset": "utf8mb4",
    "autocommit": False,
}

# 1. 범죄 통계 로드 및 국내(외국 제외) Long 형태 변환
crime_raw = pd.read_csv(crime_path, encoding="cp949")
crime_raw.columns = [c.strip() for c in crime_raw.columns]
crime_raw["범죄대분류"] = crime_raw["범죄대분류"].astype(str).str.strip()
crime_raw["범죄중분류"] = crime_raw["범죄중분류"].astype(str).str.strip()

melted = crime_raw.melt(
    id_vars=["범죄대분류", "범죄중분류"],
    var_name="지역전체",
    value_name="발생건수",
)
melted["지역전체"] = melted["지역전체"].astype(str).str.strip()
split_reg = melted["지역전체"].str.split(" ", n=1, expand=True)
melted["시도_구분"] = split_reg[0].str.strip()
melted["시군구_상세"] = split_reg[1].fillna("").str.strip()
melted["발생건수"] = (
    pd.to_numeric(melted["발생건수"], errors="coerce").fillna(0).astype(int)
)

# 국내 지역만 필터링
domestic_crime = melted[melted["시도_구분"] != "외국"].copy()
domestic_regions = sorted(domestic_crime["지역전체"].unique())

# 2. CCTV 로드 및 국내 지역별 카메라 대수 집계
cctv_raw = pd.read_csv(cctv_path, encoding="cp949", low_memory=False)
cctv_raw["카메라대수_num"] = pd.to_numeric(
    cctv_raw["카메라대수"], errors="coerce"
).fillna(0)

sido_map = {
    "서울특별시": "서울",
    "서울": "서울",
    "부산광역시": "부산",
    "부산": "부산",
    "대구광역시": "대구",
    "대구": "대구",
    "인천광역시": "인천",
    "인천": "인천",
    "광주광역시": "광주",
    "광주": "광주",
    "대전광역시": "대전",
    "대전": "대전",
    "울산광역시": "울산",
    "울산": "울산",
    "세종특별자치시": "세종시",
    "세종시": "세종시",
    "경기도": "경기도",
    "경기": "경기도",
    "강원특별자치도": "강원도",
    "강원도": "강원도",
    "강원": "강원도",
    "충청북도": "충북",
    "충북": "충북",
    "충청남도": "충남",
    "충남": "충남",
    "전북특별자치도": "전북",
    "전라북도": "전북",
    "전북": "전북",
    "전라남도": "전남",
    "전남": "전남",
    "경상북도": "경북",
    "경북": "경북",
    "경상남도": "경남",
    "경남": "경남",
    "제주특별자치도": "제주",
    "제주도": "제주",
    "제주": "제주",
}
gwangju_gus = {"동구", "서구", "남구", "북구", "광산구"}
code_to_region = {
    3030000: "서울 성동구",
    3240000: "서울 강동구",
    3290000: "부산 진구",
    3310000: "부산 남구",
    3690000: "울산 중구",
    4231000: "강원도 속초시",
    4311000: "강원도 화천군",
}


def resolve_cctv_region(row):
  code = row.get("개방자치단체코드")
  if pd.notna(code):
    try:
      c_int = int(code)
      if c_int in code_to_region:
        return code_to_region[c_int]
    except (ValueError, TypeError):
      pass

  addr = row.get("소재지도로명주소")
  if pd.isna(addr) or not str(addr).strip():
    addr = row.get("소재지지번주소")
  addr_str = str(addr or "").strip()
  if not addr_str:
    return ""

  toks = addr_str.split()
  t0 = toks[0].strip()
  t1 = toks[1].strip() if len(toks) > 1 else ""
  t2 = toks[2].strip() if len(toks) > 2 else ""

  if t0 == "산청군":
    return "경남 산청군"
  if t0 == "전남광주통합특별시":
    if t1 in gwangju_gus:
      return f"광주 {t1}"
    return f"전남 {t1}" if t1 else ""

  sido = sido_map.get(t0, "")
  if not sido:
    return ""
  if sido == "세종시" or t1 == "세종시":
    return "세종시"
  if not t1:
    return ""

  for g in ["중구", "동구", "서구", "남구", "북구"]:
    if t1.startswith(g) and t1 != g and not t1.endswith("동"):
      t1 = g
      break

  if t1.endswith("시") and t2.endswith("구"):
    return f"{sido} {t1}"
  return f"{sido} {t1}"


cctv_raw["지역전체"] = cctv_raw.apply(resolve_cctv_region, axis=1)
cctv_by_reg = (
    cctv_raw[cctv_raw["지역전체"].isin(domestic_regions)]
    .groupby("지역전체", as_index=False)["카메라대수_num"]
    .sum()
)
cctv_map = {
    r.지역전체: int(round(r.카메라대수_num))
    for r in cctv_by_reg.itertuples(index=False)
}

# 3. MariaDB 적재
conn = pymysql.connect(**db_config)
try:
  with conn.cursor() as cur:
    # A. region_info 적재
    reg_rows = domestic_crime[
        ["지역전체", "시도_구분", "시군구_상세"]
    ].drop_duplicates()
    cur.executemany(
        """
            INSERT IGNORE INTO region_info (region_full, sido_name, sigungu_detail)
            VALUES (%s, %s, %s)
            """,
        list(reg_rows.itertuples(index=False, name=None)),
    )

    # B. crime_category 적재
    cat_rows = domestic_crime[["범죄대분류", "범죄중분류"]].drop_duplicates()
    cur.executemany(
        """
            INSERT IGNORE INTO crime_category (major_category, minor_category)
            VALUES (%s, %s)
            """,
        list(cat_rows.itertuples(index=False, name=None)),
    )

    # C. ID 조회
    cur.execute("SELECT region_id, region_full FROM region_info")
    reg_id_map = {full: rid for rid, full in cur.fetchall()}

    cur.execute(
        "SELECT category_id, major_category, minor_category FROM crime_category"
    )
    cat_id_map = {(maj, min_): cid for cid, maj, min_ in cur.fetchall()}

    # D. crime_stats 적재
    stat_rows = []
    for r in domestic_crime.itertuples(index=False):
      rid = reg_id_map.get(r.지역전체)
      cid = cat_id_map.get((r.범죄대분류, r.범죄중분류))
      if rid is not None and cid is not None:
        stat_rows.append((rid, cid, int(r.발생건수)))

    cur.executemany(
        """
            INSERT INTO crime_stats (region_id, category_id, occurrence_cnt)
            VALUES (%s, %s, %s)
            ON DUPLICATE KEY UPDATE occurrence_cnt = VALUES(occurrence_cnt)
            """,
        stat_rows,
    )

    # E. region_cctv_stats 적재
    cctv_rows = [
        (rid, cctv_map.get(full, 0)) for full, rid in reg_id_map.items()
    ]
    cur.executemany(
        """
            INSERT INTO region_cctv_stats (region_id, cctv_cnt)
            VALUES (%s, %s)
            ON DUPLICATE KEY UPDATE cctv_cnt = VALUES(cctv_cnt)
            """,
        cctv_rows,
    )

  conn.commit()
  print(
      f"재생성 DB 적재 완료! (지역 {len(reg_id_map)}개, 분류"
      f" {len(cat_id_map)}개, 범죄통계 {len(stat_rows)}행, CCTV"
      f" {len(cctv_rows)}행)"
  )
except Exception as e:
  conn.rollback()
  print("오류 발생으로 롤백됨:", e)
  raise
finally:
  conn.close()

재생성 DB 적재 완료! (지역 229개, 분류 38개, 범죄통계 8702행, CCTV 229행)
